# Jira Theme / Value Stream Coverage Audit

This notebook treats linked **Theme** tickets as the value stream signal.

## What it does

For Jira tickets created on or after a chosen date:

1. Fetch source tickets from Neo4j
2. Read linked `GROUP-*` keys from `inwardIssues`
3. Resolve those linked keys to actual Jira nodes
4. Keep only linked nodes where `issueType = "Theme"`
5. Count, per ticket:
   - whether the ticket has at least one linked Theme
   - how many linked Themes it has
   - which Theme keys and names are linked

## Current assumption

A ticket is counted as having a valid value stream if it has **at least one linked `GROUP-*` node whose `issueType` is `Theme`**.

If your business logic changes later, you can replace this with a stricter mapping step.

In [ ]:
%pip install -q neo4j pandas

In [ ]:
from neo4j import GraphDatabase
import pandas as pd
from datetime import datetime

## Configuration

Update the Neo4j credentials and business filters below before running.

In [ ]:
# -----------------------------
# Neo4j connection settings
# -----------------------------
NEO4J_URI = "bolt://localhost:7687"
NEO4J_USER = "neo4j"
NEO4J_PASSWORD = "YOUR_PASSWORD"   # <-- change this
NEO4J_DATABASE = "neo4j"

# -----------------------------
# Business filters
# -----------------------------
SINCE_DATE = "2023-01-01"              # tickets created on or after this date
SOURCE_ISSUE_TYPE = "Engagement Request"  # set to None to include all issue types
THEME_ISSUE_TYPE = "Theme"             # linked GROUP ticket type treated as valid VS

# Convert date to epoch seconds because the graph stores creationDateEpoch
SINCE_EPOCH = int(datetime.fromisoformat(SINCE_DATE + "T00:00:00+00:00").timestamp())

print("SINCE_DATE :", SINCE_DATE)
print("SINCE_EPOCH:", SINCE_EPOCH)

## Helper: Neo4j runner

In [ ]:
driver = GraphDatabase.driver(
    NEO4J_URI,
    auth=(NEO4J_USER, NEO4J_PASSWORD),
)

def run_query(cypher: str, params: dict | None = None) -> pd.DataFrame:
    """
    Run a Cypher query and return the results as a pandas DataFrame.
    """
    with driver.session(database=NEO4J_DATABASE) as session:
        result = session.run(cypher, params or {})
        return pd.DataFrame([record.data() for record in result])

print("Connected to Neo4j driver.")

## Optional sanity checks

In [ ]:
# How many JIRA nodes exist?
jira_count_df = run_query("""
MATCH (n:JIRA)
RETURN count(n) AS jira_count
""")
jira_count_df

In [ ]:
# Quick look at recent source tickets
issue_type_filter = ""
params = {"since_epoch": SINCE_EPOCH}

if SOURCE_ISSUE_TYPE:
    issue_type_filter = "AND n.issueType = $source_issue_type"
    params["source_issue_type"] = SOURCE_ISSUE_TYPE

recent_source_df = run_query(f"""
MATCH (n:JIRA)
WHERE n.creationDateEpoch >= $since_epoch
  {issue_type_filter}
RETURN
    n.key AS ticket_key,
    n.issueType AS issue_type,
    n.creationDate AS creation_date,
    n.creationDateEpoch AS creation_epoch,
    n.summary AS summary,
    n.inwardIssues AS inward_issues
ORDER BY n.creationDateEpoch DESC
LIMIT 20
""", params)

recent_source_df

## Main fetch function

This is the core logic:

- Start from source tickets after the chosen date
- Read `inwardIssues`
- Keep only entries that start with `GROUP-`
- Resolve those keys to actual Jira nodes
- Keep only linked nodes where `issueType = "Theme"`

In [ ]:
def fetch_ticket_theme_links(
    since_epoch: int,
    source_issue_type: str | None = None,
    theme_issue_type: str = "Theme"
) -> pd.DataFrame:
    """
    Returns one row per source ticket x linked GROUP key.

    If the linked GROUP node exists and has issueType == theme_issue_type,
    its details appear in the theme_* columns.
    Otherwise theme_* columns will be null for that row.
    """
    issue_type_filter = ""
    if source_issue_type:
        issue_type_filter = "AND t.issueType = $source_issue_type"

    cypher = f"""
    MATCH (t:JIRA)
    WHERE t.creationDateEpoch >= $since_epoch
      {issue_type_filter}

    WITH
        t,
        [k IN coalesce(t.inwardIssues, []) WHERE k STARTS WITH "GROUP-"] AS candidate_group_keys

    UNWIND CASE
        WHEN size(candidate_group_keys) = 0 THEN [NULL]
        ELSE candidate_group_keys
    END AS group_key

    OPTIONAL MATCH (g:JIRA {{key: group_key}})
    WHERE g.issueType = $theme_issue_type

    RETURN
        t.key AS ticket_key,
        t.issueType AS ticket_issue_type,
        t.summary AS ticket_summary,
        t.creationDate AS ticket_creation_date,
        t.creationDateEpoch AS ticket_creation_epoch,

        group_key AS linked_group_key,
        g.key AS theme_key,
        g.issueType AS theme_issue_type,
        g.summary AS theme_summary,
        g.creationDate AS theme_creation_date
    ORDER BY ticket_creation_epoch DESC, ticket_key, linked_group_key
    """

    params = {
        "since_epoch": since_epoch,
        "theme_issue_type": theme_issue_type,
    }
    if source_issue_type:
        params["source_issue_type"] = source_issue_type

    return run_query(cypher, params)

## Run the extraction

In [ ]:
raw_links_df = fetch_ticket_theme_links(
    since_epoch=SINCE_EPOCH,
    source_issue_type=SOURCE_ISSUE_TYPE,
    theme_issue_type=THEME_ISSUE_TYPE,
)

print("Raw row count      :", len(raw_links_df))
print("Distinct ticket count:", raw_links_df["ticket_key"].nunique() if not raw_links_df.empty else 0)

raw_links_df.head(20)

In [ ]:
# Random sample for manual inspection
if not raw_links_df.empty:
    display(raw_links_df.sample(min(10, len(raw_links_df)), random_state=42))
else:
    print("No rows returned.")

## Aggregate to one row per source ticket

In [ ]:
def unique_non_null(values):
    return sorted({
        v for v in values
        if pd.notna(v) and str(v).strip() != ""
    })

if raw_links_df.empty:
    ticket_summary_df = pd.DataFrame(columns=[
        "ticket_key",
        "ticket_issue_type",
        "ticket_summary",
        "ticket_creation_date",
        "ticket_creation_epoch",
        "theme_keys",
        "theme_names",
        "linked_group_keys",
        "total_vs_in_ticket",
        "has_valid_vs",
    ])
else:
    ticket_summary_df = (
        raw_links_df
        .groupby(
            [
                "ticket_key",
                "ticket_issue_type",
                "ticket_summary",
                "ticket_creation_date",
                "ticket_creation_epoch",
            ],
            dropna=False,
            as_index=False
        )
        .agg(
            theme_keys=("theme_key", unique_non_null),
            theme_names=("theme_summary", unique_non_null),
            linked_group_keys=("linked_group_key", unique_non_null),
        )
    )

    ticket_summary_df["total_vs_in_ticket"] = ticket_summary_df["theme_keys"].apply(len)
    ticket_summary_df["has_valid_vs"] = ticket_summary_df["total_vs_in_ticket"] > 0

    ticket_summary_df = ticket_summary_df.sort_values(
        by=["ticket_creation_epoch", "ticket_key"],
        ascending=[False, True]
    ).reset_index(drop=True)

ticket_summary_df.head(20)

## Summary statistics

In [ ]:
total_tickets = len(ticket_summary_df)
valid_tickets = int(ticket_summary_df["has_valid_vs"].sum()) if total_tickets else 0
invalid_tickets = total_tickets - valid_tickets

summary_stats = pd.DataFrame([{
    "since_date": SINCE_DATE,
    "source_issue_type": SOURCE_ISSUE_TYPE if SOURCE_ISSUE_TYPE else "ALL",
    "theme_issue_type": THEME_ISSUE_TYPE,
    "total_tickets": total_tickets,
    "valid_tickets_with_theme": valid_tickets,
    "tickets_without_theme": invalid_tickets,
    "coverage_pct": round((valid_tickets / total_tickets) * 100, 2) if total_tickets else 0.0,
}])

summary_stats

## Positive and negative examples

In [ ]:
# Tickets that have at least one valid Theme
ticket_summary_df[ticket_summary_df["has_valid_vs"]].head(20)

In [ ]:
# Tickets that do not have any valid Theme
ticket_summary_df[~ticket_summary_df["has_valid_vs"]].head(20)

## Final business-facing output

In [ ]:
final_df = ticket_summary_df[[
    "ticket_key",
    "ticket_issue_type",
    "ticket_summary",
    "ticket_creation_date",
    "has_valid_vs",
    "total_vs_in_ticket",
    "theme_keys",
    "theme_names",
]]

final_df.head(20)

## Helpful extra queries

Use these when you want to inspect edge cases.

In [ ]:
# 1) Inspect a single ticket and its inward issues
SAMPLE_TICKET_KEY = "IDMT-19761"   # change if needed

single_ticket_df = run_query("""
MATCH (n:JIRA)
WHERE n.key = $ticket_key
RETURN
    n.key AS ticket_key,
    n.issueType AS issue_type,
    n.creationDate AS creation_date,
    n.summary AS summary,
    n.inwardIssues AS inward_issues,
    n.inwardIssuesMetaData AS inward_issues_metadata
""", {"ticket_key": SAMPLE_TICKET_KEY})

single_ticket_df

In [ ]:
# 2) Inspect a single GROUP ticket
SAMPLE_GROUP_KEY = "GROUP-22223"   # change if needed

single_group_df = run_query("""
MATCH (n:JIRA)
WHERE n.key = $group_key
RETURN
    n.key AS group_key,
    n.issueType AS issue_type,
    n.creationDate AS creation_date,
    n.summary AS summary,
    n.outwardIssues AS outward_issues,
    n.outwardIssuesMetaData AS outward_issues_metadata
""", {"group_key": SAMPLE_GROUP_KEY})

single_group_df

## Export CSV outputs

In [ ]:
final_df.to_csv("ticket_theme_summary.csv", index=False)
raw_links_df.to_csv("ticket_theme_raw_links.csv", index=False)
summary_stats.to_csv("ticket_theme_stats.csv", index=False)

print("Saved:")
print(" - ticket_theme_summary.csv")
print(" - ticket_theme_raw_links.csv")
print(" - ticket_theme_stats.csv")

## Close the driver when done

In [ ]:
# Run this when you are fully done with the notebook
# driver.close()